#  AI Voice Training Studio (RVC)
Notebook ini berisi panduan langkah-demi-langkah beserta baris kode untuk melakukan **training (pelatihan) model suara RVC** menggunakan dataset rekaman suara Anda sendiri atau orang lain.

> **⚠️ PERINGATAN PENTING (Kebutuhan Sistem):** 
> Training AI sangat berat dan membutuhkan **GPU NVIDIA** (VRAM minimal 6GB, disarankan 8GB ke atas). 
> 
> **Jika komputer Anda tidak memiliki GPU yang memadai, SANGAT DISARANKAN untuk meng-upload file Notebook (`RVC_Training.ipynb`) ini ke [Google Colab](https://colab.research.google.com/)** lalu jalankan semua selnya di sana menggunakan GPU gratis dari Google.

## 1. Persiapan Lingkungan & Unduh Engine Training
Kita akan menggunakan repositori resmi RVC untuk melakukan proses training.

In [ ]:
# Clone Repositori RVC Project
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc_training_workspace

# Masuk ke direktori hasil clone
%cd rvc_training_workspace

In [ ]:
# Install semua library dan dependencies yang dibutuhkan
!pip install -r requirements.txt
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

## 2. Download Pre-Trained Base Models
Model dasar ini diperlukan sebagai fondasi. Model ini sudah dilatih dengan ribuan jam suara manusia agar AI mengerti bagaimana manusia berbicara dan bernyanyi sebelum Anda mengajarinya suara spesifik.

In [ ]:
import os
import urllib.request

def download_file(url, path):
    if not os.path.exists(path):
        print(f"Mengunduh {path}...")
        urllib.request.urlretrieve(url, path)
        print("✅ Selesai!")
    else:
        print(f"✅ File {path} sudah ada.")

# Folder untuk model v2
os.makedirs("pretrained_v2", exist_ok=True)

# Pretrained F0 (Vokal + Pitch)
download_file("https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth", "pretrained_v2/f0G40k.pth")
download_file("https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth", "pretrained_v2/f0D40k.pth")

# HuBERT (Fitur Pengenalan Kata/Pengucapan)
download_file("https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt", "hubert_base.pt")

# RMVPE (Algoritma ekstraksi nada/pitch terbaik saat ini)
download_file("https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt", "rmvpe.pt")

## 3. Persiapan Dataset Suara Anda
Tahap ini paling penting!
1. Rekam atau kumpulkan suara Anda / target yang ingin dilatih (10 - 20 menit sudah sangat bagus).
2. Pastikan file suaranya **bersih** (tidak ada suara musik, echo, noise, atau vokal orang lain).
3. Letakkan semua file audio (WAV) tersebut ke dalam folder yang Anda tentukan di sel bawah ini.

In [ ]:
EXPERIMENT_NAME = "suara_saya_v1" # Nama model baru Anda
DATASET_PATH = "../dataset_suara" # Path folder tempat Anda menaruh file rekaman

os.makedirs(DATASET_PATH, exist_ok=True)
print(f"Folder dataset telah dibuat di: {os.path.abspath(DATASET_PATH)}")
print("SILAKAN MASUKKAN file rekaman (.wav / .mp3) ke dalam folder tersebut sebelum melanjutkan.")

## 4. Preprocessing Data (Ekstraksi Fitur)
AI akan memotong audio Anda, mengekstrak nada dasar (F0), dan mengumpulkan fitur vokal.

In [ ]:
# 1. Memotong dan menyiapkan dataset
!python infer/modules/train/preprocess.py {DATASET_PATH} 40000 2 ./logs/{EXPERIMENT_NAME} False

# 2. Ekstraksi Pitch (F0) dengan RMVPE (Paling stabil dan bersih)
!python infer/modules/train/extract/extract_f0_print.py ./logs/{EXPERIMENT_NAME} 2 rmvpe

# 3. Ekstraksi Fitur Vokal (HuBERT)
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 0 ./logs/{EXPERIMENT_NAME} v2

## 5. Mulai Training (Pelatihan)
Ini adalah proses utamanya. Proses ini akan berlangsung cukup lama.
- `SAVE_FREQUENCY`: Model akan disimpan setiap berapa epoch.
- `TOTAL_EPOCH`: Jumlah siklus pembelajaran. Disarankan minimal **100**, dan maksimal **300** untuk hasil terbaik agar tidak *overfit*.
- `BATCH_SIZE`: Jumlah memori GPU yang digunakan (Gunakan 8 jika VRAM Anda 8GB+, gunakan 4 jika VRAM 6GB).

In [ ]:
SAVE_FREQUENCY = 10
TOTAL_EPOCH = 150 
BATCH_SIZE = 8

!python infer/modules/train/train.py -e {EXPERIMENT_NAME} -sr 40k -f0 1 -bs {BATCH_SIZE} -te {TOTAL_EPOCH} -se {SAVE_FREQUENCY} -v v2 -g 0

## 6. Training Index File
Langkah terakhir. Index file berfungsi agar AI bisa mengingat kembali cengkok dan keunikan pengucapan dari suara asli Anda.

In [ ]:
!python infer/modules/train/train_index.py -exp {EXPERIMENT_NAME} -v v2

## 7. Evaluasi Visual (TensorBoard)
Jalankan sel di bawah ini **saat proses training (Langkah 5) sedang berlangsung** (misal di tab baru) atau **setelah selesai** untuk melihat grafik *Loss*.
Pastikan grafik secara umum menurun seiring berjalannya waktu, yang menandakan AI semakin pintar meniru suara Anda.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

##  Selesai! Mengambil Model
Jika semua sel di atas selesai tanpa *error*:
1. **File Model Utama (.pth)** Anda bisa ditemukan di: `rvc_training_workspace/assets/weights/suara_saya_v1.pth`
2. **File Index (.index)** Anda bisa ditemukan di: `rvc_training_workspace/logs/suara_saya_v1/added_xxxx.index`

Sekarang, Anda bisa memindahkan kedua file tersebut ke dalam folder `models/` di project **AI Cover Studio** Anda (di aplikasi Web) lalu menggunakannya seperti model biasa!